# Coding Session #9

---

## Today's session

Today's session is structured to teach more complex tasks. We will explore:

1. Casual inference Vs. prediction
2. Data preparation and regression  
3. Comparing predicted and actual outcomes
4. Confusion matrix
5. Model metrics

### 0.1 Environment preparation

We begin by loading the libraries we’ll need. In Python, libraries are like toolkits: they extend the language with specialized functions.

- **pandas (pd)** is our main tool for working with tabular data. It introduces the DataFrame, which lets us manipulate datasets in a way that feels natural if you’ve used Excel or R.
- **NumPy (np)** provides the numerical backbone. It gives us arrays and fast mathematical functions, which pandas actually uses under the hood.
- **Seaborn (sns)** builds on top of Matplotlib to create a vast range of plots and visualization.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm

## 1. Causal Inference vs. Prediction

*Mindset:* In predictive modeling, the focus shifts away from estimating a single coefficient precisely to maximizing the accuracy of predicted outcomes.

| **Aspect**                | **Causal Inference**                                                                          | **Prediction / Machine Learning**                                                    |
| ------------------------- | --------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------ |
| **Primary Goal**          | Cause–effect relationships                        | Good accuracy in forecasting or classification           |
| **Research Design**       | Experiments | Training and validation on existing data     |
| **Typical Tools**         | **Y on D** to estimate **β**                                    | Machine learning |
| **Quantity of Interest**  | The **causal effect (β)**                                                                     | The **predicted value (Ŷ)**                                                          |
| **Statistical Challenge** | What would have happened under counterfactual                              | Generalizing accurately to unseen data                                               |
| **Evaluation Criterion**  | Internal validity, unbiased estimation                                                        | Out-of-sample predictive performance                     |


Before delving into machine learning, it is important to review the fundamentals of regression analysis. There are various types of regression models that can be applied to data, but for the purpose of this exercise, we will focus on logistic regression—a model particularly useful when the dependent variable (Y) is binary, such as whether an individual turned out to vote or not.

Our goal here is to understand how to run and interpret a logistic regression model, with specific attention to how we interpret the coefficients from our previous experiment on robocalls.

### Citation

Kling, Daniel T., and Thomas Stratmann. 2023. *Large-Scale Evidence for the Effectiveness of Partisan GOTV Robo Calls*. *Journal of Experimental Political Science*. https://doi.org/10.1017/XPS.2022.16

#### Codebook


## Codebook

| Variable   | Type        | Description                                                                 |
|------------|-------------|-----------------------------------------------------------------------------|
| **age**    | Continuous  | Age of registered voter at time of experiment|
| **income** | Continuous | Estimated household income based on external vendor data.                   |
| **male**   | Binary      | Gender of voter.                                                            |
| **svh**    | Binary      | Indicator for single-voter household                                     |
| **gen2012**| Binary      | Indicator for having voted in the 2012 General Election.                    |
| **treatment** | Binary   | Indicator for assignment to any treatment condition.                        |
| **group**     | Categorical   | Indicator for assignment to 1-call, 3-call, and 6-call treatment group.  |
| **voted** | Binary       | Indicator of having voted in 2014 General Election.                       |


In [ ]:
df_kling = pd.read_csv("https://raw.githubusercontent.com/albertostefanelli/DSPC_coding_sessions/refs/heads/main/codingsession08/data/kling_stratmann_subset.csv")

## 2. Data preparation and regression  


### Variable types

Let's take a lookat the dataset and see what we have. Regression models in Python do not take `object` variables. So we need either to exclude them or create a series of dummies to include in the regression (don't worry about it for the moment, we will see how to do it in the next lab).

In [ ]:
print(df_kling.dtypes)

In [ ]:
df_kling.info()

Let's first drop our group variable. This variable refers to the treatement arm that the respondent was assigned (1-call, 3-call, 6-call).

In [ ]:
kling_sub = df_kling[['age', 'income', 'male', 'svh', 'gen2012', 'treatment', 'voted']].copy()

### Missing data (NAs)

Next, let's check whether our dataset contains any missing values (NAs). In Python, when running regression models missing values can cause errors or silently drop observations from the analysis. Therefore, it is good practice to inspect and handle NAs before estimation.

Using pandas, we can easily identify missing data with functions like `df.info()` or `df.isna().sum()`, and if necessary, remove incomplete observations using `df.dropna()`. This ensures that our model is fitted on a clean, consistent dataset and that any issues related to missing values are addressed upfront.

In [ ]:
kling_sub.isna().sum()

In [ ]:
kling_sub_c = kling_sub.dropna().copy()

In [ ]:
kling_sub_c.isna().sum()

### DV and IVs

Next, let's create two separate objects: one for our **dependent variable** (which we’ll call `Y`) and one for our **independent variables** (which we’ll call `X`). This structure follows the standard format used by most Python statistical packages---such as `statsmodels`---where models are defined in the form `Y = f(X)`.

In this setup, `Y` represents the **outcome** we aim to explain or predict (for example, whether a voter turned out to vote), while `X` contains the **predictor variables** that may influence that outcome (such as age, income, gender, and previous voting behavior).

Separating `Y` and `X` not only clarifies the modeling process but also makes it easier to pass these objects directly into regression functions like `sm.Logit(Y, X)` or `sm.OLS(Y, X)`.


In [ ]:
X = kling_sub_c[['age', 'income', 'male', 'svh', 'gen2012', 'treatment']].copy()
Y = kling_sub_c['voted']

Next, we need to add an intercept (also called a constant term) to our set of independent variables, X. The intercept represents the expected value of the dependent variable, Y, when all predictors are equal to zero—it serves as the baseline level of the outcome.

In statsmodels, intercepts are not added automatically, so we must include one manually. This ensures that the regression model can estimate both the intercept and the slope coefficients correctly. Failing to include a constant would force the regression line to pass through the origin, which can bias results if the true relationship does not.

In [ ]:
X['intercept'] = 1

In [ ]:
X.head()

In [ ]:
Y.head()

### Fitting the model and interpret the results

Finally we fit a logistic model to the data


In [ ]:
logit_model = sm.Logit(Y,X)
result = logit_model.fit()
print(result.summary2())

Congratulations! You run your first regresison model! A logistic regression models the log odds of the dependent variable (usually a binary outcome, e.g., 1 = voted, 0 = not voted) as a linear combination of predictors. Each coefficient represents the change in the log-odds of the outcome associated with a one-unit increase in the predictor, holding all others constant. To interpret in terms of odds ratio, you exponentiate the coefficient such that $e^{\beta_i} = \text{odds ratio}$

- If $e^{\beta_i} > 1$: the predictor **increases** the odds of the outcome.  
- If $e^{\beta_i} < 1$: the predictor **decreases** the odds of the outcome.  
- If $e^{\beta_i} = 1$: the predictor has **no effect** on the odds.




In [ ]:
# odds for age and svh
np.exp(0.0298), np.exp(-0.4595)

**Continuous Variable — age**

- Interpretation: Each unit increase in the variable (1 year of age) changes the log-odds of the outcome by 0.0298.
- Converted to odds: exp(0.029) =1.03, meaning the odds increase by 3% per year on the odds scale
- This effect is linear and incremental — every extra year adds the same proportional increase in odds.

**Dummy Variable — SVH**
- Interpretation: The coefficient of −0.4595 means that being in the SVH group decreases the log-odds of the outcome by 0.4595 compared to the reference group (those with svh = 0).
- Converted to odds: exp(−0.4595)=0.63, meaning that being in the SVH group reduces the odds of the outcome by 1-0.63 = 37% relative to the reference group.
- This effect is categorical — it represents a one-time shift in odds between two groups rather than a continuous or incremental change.

### What about the coefficent for income?

- Interpretation: The coefficient for income appears as 0.0000, but it’s statistically significant, meaning the true effect is very small per unit of income
- Regression coefficients reflect the effect of a one-unit change. A one-dollar change has little impact
- Example: If the true coefficient were 0.00004, then a 10000 increase gives 0.00004 × 10000= 0.4 OR exp(0.4) = 1.49 a 49% increase in odds
-Conclusion:  The effect may look negligible only because the variable’s scale is large. Rescaling makes coefficients interpretable and highlights meaningful effects.

In [ ]:
X["income"] =  X["income"] / 10000

In [ ]:
logit_model = sm.Logit(Y,X)
result = logit_model.fit()
print(result.summary2())

### Comparing predicted and actual outcomes using predicted outcome

- In regression, the model estimates a mathematical relationship between predictors (X) and an outcome (y).
- Once the model is fitted, we can use it to predict the expected value of the outcome for given values of the predictors.
- In regression, the model estimates a mathematical relationship between predictors (X) and an outcome (y) for each observation (e.g., voter) in the dataset

In [ ]:
y_pred = result.predict(X)
y_pred.head()

### A good predictive model - comparing predicted and actual outcomes

A good predictive model should assign **high probabilities** to cases that actually occurred (1s) and **low probabilities** to cases that did not (0s).  

The central question is:  

> Do higher predicted probabilities from the model actually correspond to higher rates of people who voted?

To evaluate this, we can plot the relationship between the model’s predicted probabilities and the observed outcomes. This visual check helps determine whether the model’s predicted probabilities align with real-world frequencies — in other words, whether the model is **well calibrated**.

Technically, this can be achieved by:  
- **Fitting the model** and calculating each observation’s **predicted probability** (`y_pred`).  
- **Dividing** the predicted probabilities into bins (e.g., ten bins ranging from 0–10%, 10–20%, … up to 90–100%).  
- **Computing** the **average actual turnout** (the mean of the binary outcome) within each bin, showing the proportion of individuals who actually voted.  
- **Plotting** these results as a **bar chart**, with predicted probability bins on the x-axis and average observed turnout on the y-axis.

If the model is accurate, the bars should show a clear upward trend: groups with higher predicted probabilities should exhibit higher actual voting rates, indicating that the model successfully captures the



In [ ]:
kling_sub_c['y_pred'] = y_pred

kling_sub_c['bin'] = pd.cut(x = y_pred, bins = [0,
                                                0.1,
                                                0.2,
                                                0.3,
                                                0.4,
                                                0.5,
                                                0.6,
                                                0.7,
                                                0.8,
                                                0.9,
                                                1.0],
  labels = ['0-10%',
            '10-20%',
            '20-30%',
            '30-40%',
            '40-50%',
            '50-60%',
            '60-70%',
            '70-80%',
            '80-90%',
            '90-100%'])

df = (
    kling_sub_c
    .groupby('bin', observed=False)['voted']
    .mean()
    .dropna()
    .reset_index()
)

barpl = sns.barplot(
    data=df,
    x='bin',
    y='voted'
)

barpl.set(
    xlabel='Predicted probability of voting',
    ylabel='Average turnout (observed)',
    title='Likely voter model compared to actual turnout'
)

barpl.tick_params(axis='x', labelrotation=45)

That bar chart visually compares the **predicted probabilities of voting** generated by the model to the **actual turnout rates** observed in 2014.

Here’s how to interpret it:

- Each bar represents a **bin of predicted probabilities** (for example, 0–10%, 10–20%, …, 90–100%).
- The height of each bar shows the **average observed turnout** within that bin — the proportion of individuals who actually voted within that predicted probability range.

Interpretation:

- The plot shows a clear **upward trend**, meaning that as the model’s predicted probability of voting increases, the actual observed turnout also increases. This indicates that the model correctly captures the general direction of the relationship between predicted and true outcomes.
- However, the **average observed turnout** (the height of the bars) is sometimes **higher than the model’s predicted probabilities**, suggesting that the model **underpredicts** actual voting behavior. This underestimation is particularly visible in the **10–20%** and **80–90%** probability bins, where actual turnout exceeds what the model anticipated.
- Additionally, one of the upper bins (90–100%) appears to be missing, indicating that the model **never predicted anyone with near-certain probability of voting**. This implies that the model avoids assigning extreme probabilities, even to individuals who are very likely to vote.


## 4. Confusion Matrix

To formally assess how well the model predicts voting behavior, we can construct a **2×2 confusion matrix**. This matrix compares the model’s **binary predictions** (vote vs. not vote) against the **actual observed outcomes**, allowing us to evaluate how accurately the model classifies individuals into each category.

|                         | **Actual: Did Not Vote (0)** | **Actual: Voted (1)** |
|--------------------------|------------------------------|------------------------|
| **Predicted: Did Not Vote (0)** | True Negatives (TN) | False Negatives (FN) |
| **Predicted: Voted (1)**        | False Positives (FP) | True Positives (TP) |

---

### How to Construct the Confusion Matrix

1. **Fit the model and obtain predicted probabilities (`y_pred`):**  
   These probabilities represent how likely each individual is to have voted, as estimated by the logistic regression model.  

2. **Apply a classification threshold:**  
   Convert the predicted probabilities into binary predictions (`0` or `1`) using a threshold value.  
   - For example, a **0.5 threshold** means:  
     - If `y_pred > 0.5`, classify as a predicted voter (`1`).  
     - If `y_pred ≤ 0.5`, classify as a predicted non-voter (`0`).  
   - This threshold is **arbitrary** and can be adjusted depending on the research goal (e.g., increasing recall vs. precision).

3. **Calculate the four possible outcomes:**  
   - **True Positives (TP):** Predicted voter, actually voted.  
   - **True Negatives (TN):** Predicted non-voter, actually did not vote.  
   - **False Positives (FP):** Predicted voter, actually did not vote.  
   - **False Negatives (FN):** Predicted non-voter, actually voted.

4. **Fill in the matrix with counts:**  
   By comparing predicted labels (`pred_vote`) with actual voting outcomes (`voted14`), you can compute the number of observations that fall into each of the four cells.


In [ ]:
# Step 1: Classify predictions using a 0.5 threshold
kling_sub_c['pred_vote'] = np.where(y_pred > 0.5, 1, 0)

# Step 2: Calculate confusion matrix components
# True Negative (TN): predicted 0, actual 0
tn = len(kling_sub_c.loc[(kling_sub_c['pred_vote'] == 0) & (kling_sub_c['voted'] == 0)])

# False Positive (FP): predicted 1, actual 0
fp = len(kling_sub_c.loc[(kling_sub_c['pred_vote'] == 1) & (kling_sub_c['voted'] == 0)])

# False Negative (FN): predicted 0, actual 1
fn = len(kling_sub_c.loc[(kling_sub_c['pred_vote'] == 0) & (kling_sub_c['voted'] == 1)])

# True Positive (TP): predicted 1, actual 1
tp = len(kling_sub_c.loc[(kling_sub_c['pred_vote'] == 1) & (kling_sub_c['voted'] == 1)])

# Step 3: Display confusion matrix counts
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

# Create a summary DataFrame for display
conf_matrix = pd.DataFrame({
    '': ['Predicted: Did Not Vote (0)', 'Predicted: Voted (1)'],
    'Actual: Did Not Vote (0)': [tn, fp],
    'Actual: Voted (1)': [fn, tp]
})

conf_matrix


## 5. Model metrics

From these measures, we derive performance metrics (i.e., accuracy, precision, recall) to assess how well the model performs.

**1. Accuracy**

$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$

**What it measures:**  
The proportion of all predictions that the model got right — both voters and non-voters.

**Intuition:**  
Accuracy answers: *“Out of everyone, what fraction did the model classify correctly?”*

**Why it matters:**  
It provides an overall sense of performance, but can be misleading if one class dominates (e.g., if most people didn’t vote).

**2. Precision**

$\text{Precision} = \frac{TP}{TP + FP}$

**What it measures:**  
Of all the people the model predicted *would vote*, how many actually did?

**Intuition:**  
Precision focuses on the **quality of positive predictions**, emphasizing the avoidance of false positives.  
*“When the model says someone will vote, how often is that correct?”*

**Why it matters:**  
Important when false positives are costly — for example, when we don’t want to incorrectly classify non-voters as voters.

**3. Recall (Sensitivity or True Positive Rate)**

$\text{Recall} = \frac{TP}{TP + FN}$

**What it measures:**  
Of all the people who actually voted, how many did the model correctly identify?

**Intuition:**  
Recall emphasizes the model’s ability to **capture true positives**, reducing false negatives.  
*“Out of all true voters, how many did the model successfully identify?”*

**Why it matters:**  
Important when missing positive cases is costly — for example, if identifying every likely voter is crucial.

---

### Why We Calculate All Three

Each metric highlights a different aspect of model performance:

| Metric | Focus | Sensitive to | Best for |
|---------|--------|--------------|----------|
| **Accuracy** | Overall correctness | Class imbalance | Overall performance |
| **Precision** | Correctness of positive predictions | False positives | Avoiding false alarms |
| **Recall** | Coverage of actual positives | False negatives | Capturing all positives |


In [ ]:
# Step 4: Calculate key performance metrics
acc = (tp + tn) / (tp + tn + fp + fn)
prec = tp / (tp + fp)
recall = tp / (tp + fn)

print(f"Model Accuracy: {acc:.3f}")
print(f"Model Precision: {prec:.3f}")
print(f"Model Recall: {recall:.3f}")


**Interpretation:**

An accuracy of **0.701** means that the model correctly predicts whether someone voted or not about **70% of the time**. While this reflects reasonable overall performance, accuracy alone does not reveal which type of classification errors the model tends to make.

A precision of **0.664** indicates that when the model predicts someone **did vote**, it is correct about **two-thirds of the time**. In other words, roughly 66% of the individuals the model classifies as voters actually turned out to vote. This shows the model makes quite a bit pf false positive errors.

The recall of **0.835** is relatively high, meaning the model correctly identifies about **83% of all true voters**. It captures a good portion of  people who actually voted, missing only a small portion of them.


# Congratulations!

You are done with the coding session. Questions or suggestions? Email Alberto at alberto.stefanelli@yale.edu

In [ ]:
# Install requirements
!apt-get -qq update
!apt-get install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic

from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Ask for the notebook name
notebook_name = input(
    "Enter your notebook’s exact file name,\n"
    "exactly as shown in the top-left corner of the Colab page (next to the two yellow circle icons): "
)

# Build paths
input_path = f"/content/drive/MyDrive/Colab Notebooks/{notebook_name}"
output_path = input_path.replace(".ipynb", ".pdf")

# Convert to PDF
!jupyter nbconvert --to pdf "{input_path}"

# Download the PDF
files.download(output_path)